# Day 22: LangChain RAG Chains & Automated Retrieval

Welcome to Day 22 of the AI Engineering Mastery program!

Today, we are migrating from manual vector database queries to using **LangChain's specialized RAG chains**. This shift allows us to elegantly compose retrieval mechanisms with LLM generation, cleanly separating concerns and drastically reducing boilerplate code.

## Core Theory: The "Why" and "How"

### Why move away from manual retrieval?
Previously, you might have written code that explicitly queries Qdrant for vectors, formats those documents into a string, and injects them into an LLM prompt. In a production environment, this approach becomes fragile:
- **Scalability**: Adding memory, chat history, or fallback retrievers introduces complex branching.
- **Formatting**: Handling different document formats and metadata becomes tedious.
- **Observability**: Manual orchestration makes it harder to trace the lifecycle of a prompt (e.g., using LangSmith).

### How LangChain Chains Work
LangChain provides `create_retrieval_chain` and document combining chains (like `create_stuff_documents_chain`).
- **Document Chain**: Takes retrieved documents and "stuffs" them into the LLM prompt. 
- **Retrieval Chain**: Wraps the Document Chain. It takes the user's input, fetches relevant documents from the retriever, and passes them to the Document Chain.

This decouples the *fetching* of information from the *reasoning* over that information.

## Common Pitfalls in Production

1. **Missing Prompt Variables**: `create_stuff_documents_chain` requires a prompt with a `context` input variable. Omitting this or naming it differently will cause runtime validation errors.
2. **Over-Stuffing the Context Window**: The "stuff" chain dumps all retrieved documents into the prompt. If your retriever returns too many chunks, or very large chunks, you will easily hit the LLM's token limit.
3. **Synchronous Bottlenecks**: Retrieving and generating are IO-bound tasks. In high-concurrency production systems, use the asynchronous interfaces (`ainvoke`, `abatch`) to prevent blocking your application loop.
4. **Ignoring MMR (Maximal Marginal Relevance)**: Relying solely on similarity search can retrieve redundant documents. Using MMR balances relevance with diversity, ensuring the LLM sees broader context rather than repeated phrasing.

## Code Implementation & Practical Lab

**Task**: Build a fully operational RAG chain using Qdrant and LangChain. We will use OpenAI Embeddings and Chat models to construct a production-ready application.

In [ ]:
import sys
!{sys.executable} -m pip install -qU langchain langchain-core langchain-qdrant qdrant-client langchain-openai


In [ ]:
import os
from typing import List, Any

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

def setup_qdrant_vectorstore(collection_name: str = "day_22_collection") -> QdrantVectorStore:
    """
    Initializes an in-memory Qdrant client and a LangChain VectorStore.
    
    Args:
        collection_name (str): The name of the Qdrant collection.
        
    Returns:
        QdrantVectorStore: A configured LangChain vector store.
    """
    client = QdrantClient(":memory:")
    
    # OpenAI's text-embedding-3-small uses 1536 dimensions
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )
    
    return vectorstore

def populate_vectorstore(vectorstore: QdrantVectorStore, docs: List[Document]) -> None:
    """
    Populates the vector store with initial documents.
    
    Args:
        vectorstore (QdrantVectorStore): The target vector store.
        docs (List[Document]): The documents to ingest.
    """
    vectorstore.add_documents(docs)

def build_rag_chain(vectorstore: QdrantVectorStore, llm: Any) -> Any:
    """
    Constructs the end-to-end RAG chain using LangChain's built-in retrieval chain factories.
    
    Args:
        vectorstore (QdrantVectorStore): The vector store to use for retrieval.
        llm (Any): The LLM to use for generation.
        
    Returns:
        Any: A LangChain executable chain.
    """
    # 1. Configure the retriever. We use MMR (Maximal Marginal Relevance) 
    #    to fetch diverse documents, guarding against redundant context.
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    )
    
    # 2. Define the prompt template. It MUST include a {context} variable 
    #    for the stuffed documents, and an {input} variable for the user query.
    system_prompt = (
        "You are a highly capable AI assistant specializing in software engineering.\n"
        "Use the following pieces of retrieved context to answer the user's question.\n"
        "If you do not know the answer, simply state that you don't know.\n\n"
        "{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    # 3. Create the document combining chain
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    
    # 4. Create the final retrieval chain
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    
    return rag_chain

# ==========================================
# Lab Execution
# ==========================================
if __name__ == "__main__":
    # Provide your OpenAI API key to execute
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = "sk-your-key-here"  # Ensure this is set for production
        
    print("Initializing Qdrant VectorStore...")
    vs = setup_qdrant_vectorstore()
    
    # Prepare sample documents
    sample_docs = [
        Document(page_content="LangChain's create_retrieval_chain wraps a retriever and a document chain.", metadata={"source": "docs"}),
        Document(page_content="Qdrant is a fast, scalable vector search engine.", metadata={"source": "docs"}),
        Document(page_content="MMR balances relevance and diversity in search results.", metadata={"source": "docs"}),
        Document(page_content="Always type-hint your Python code for production safety.", metadata={"source": "best_practices"})
    ]
    
    print("Ingesting documents...")
    populate_vectorstore(vs, sample_docs)
    
    print("Initializing LLM...")
    llm = ChatOpenAI(model="gpt-4o-mini")
    
    print("Building RAG Chain...")
    chain = build_rag_chain(vs, llm)
    
    query = "How does the retrieval chain work in LangChain?"
    print(f"\nExecuting Query: {query}")
    
    response = chain.invoke({"input": query}) 
    print("\n--- Response ---")
    print(response["answer"])
    print("\n--- Retrieved Context Sources ---")
    for doc in response["context"]:
        print(f"- {doc.page_content}")
    print("\nLab setup completed successfully! Run with a valid OPENAI_API_KEY to test retrieval.")
